In [36]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, GenerationConfig, AutoModelForSeq2SeqLM
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import numpy as np
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim


In [37]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [38]:
# train_df = pd.read_csv('../data/train.csv')
# train_df.head()
# train_x = []
# train_y = []
# for index, row in train_df.head(10).iterrows():
#     train_x.append(row['text'])
#     train_y.append(1)

# train_dataset = Dataset.from_dict({"text": train_x, "label":train_y})

# test_x = []
# test_y = []
# for index, row in pd.read_csv('../data/comment.csv').iterrows():
#     test_x.append(row['text'])
#     test_y.append(1 if row['is_td'] else 0)

# test_dataset = Dataset.from_dict({"text": test_x, "label":test_y})

# dataset = DatasetDict({'train': train_dataset, 'eval': test_dataset.shuffle(seed=42).select(range(10)), 'test': test_dataset.shuffle(seed=42).select(range(100))})

In [39]:
df = pd.read_csv('../data/comment.csv')
df['label'] = df['is_td'].apply(lambda x: 'yes' if x == 1 else 'no')
df = df[["text", "label"]]

undersampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
X_resampled, y_resampled = undersampler.fit_resample(df[['text']], df['label'])

# Create balanced DataFrame
df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
dataset = Dataset.from_pandas(df).train_test_split(test_size=0.8, seed=42)
dataset = dataset.remove_columns(['__index_level_0__'])
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 201
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 805
    })
})

In [40]:
# import torch
# from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
# from datasets import Dataset
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# import numpy as np

# # Check if CUDA (GPU) is available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# def tokenize_function(example, tokenizer):
#     return tokenizer(example["text"], padding=True, truncation=True, max_length=512)

# def compute_metrics(p):
#     """Compute metrics like accuracy, precision, recall, and F1 score."""
#     preds = np.argmax(p.predictions, axis=1)
#     acc = accuracy_score(p.label_ids, preds)
#     precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='binary')
#     return {
#         "accuracy": acc,
#         "precision": precision,
#         "recall": recall,
#         "f1": f1
#     }

# def train_model(model_name, num_epochs, dataset):
#     """Train SATD detection model with a specified transformer model."""
    
#     GenerationConfig
#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    
#     model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
#     model.to(device)  # Ensure the model is moved to the correct device (GPU or CPU)
    
#     training_args = TrainingArguments(
#         output_dir="./cache/results",
#         num_train_epochs=num_epochs,
#         eval_strategy="epoch",  # Updated from `evaluation_strategy` to `eval_strategy`
#         save_strategy="epoch",  # Matching evaluation_strategy
#         load_best_model_at_end=True,
#         logging_dir="./cache/logs",
#         logging_steps=10,             # Log every 10 steps
#         per_device_train_batch_size=2,  # Smaller batch size for few-shot
#         per_device_eval_batch_size=2,
#         warmup_steps=500,
#         weight_decay=0.01,
#         report_to="none"  # Prevent reporting to external services like WandB
#     )
    
#     trainer = Trainer(
#         model=model, 
#         args=training_args, 
#         train_dataset=dataset['train'], 
#         eval_dataset=dataset['eval'],
#         tokenizer=tokenizer,  # Updated to tokenizer=tokenizer, as this will be deprecated in future versions
#         compute_metrics=compute_metrics
#     )
    
#     trainer.train()
#     return model, tokenizer, dataset

# def predict(model, tokenizer, texts):
#     """Make predictions using the trained model."""
#     inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)  # Move inputs to the same device
#     outputs = model(**inputs)
#     predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()  # Move outputs back to CPU for NumPy
#     return predictions

# # Example usage
# if __name__ == "__main__":
#     model_name = "roberta-base"  # Change this to switch models (e.g., "bert-base-uncased", "distilbert-base-uncased", etc.)
#     model, tokenizer, dataset = train_model(model_name,3, dataset=dataset)

#     # Test predictions on the test set only
#     test_samples = dataset['test']['text']
#     test_predictions = predict(model, tokenizer, test_samples)
    
#     test_labels = dataset['test']['label']
#     test_accuracy = accuracy_score(test_labels, test_predictions)

#     print(f"Test Accuracy: {test_accuracy}")

#     # Compute precision, recall, and F1 score for the test set
#     test_metrics = precision_recall_fscore_support(test_labels, test_predictions, average='binary')

#     print(f"Test Precision, Recall, F1: {test_metrics}")


In [41]:
class PromptTemplate:
    def __init__(self, description, example):
        self._description = description
        self._example = example

    @property
    def description(self):
        """Getter for description."""
        return self._description

    @property
    def example(self):
        """Getter for example."""
        return self._example

    def __repr__(self):
        return f"PromptTemplate(description='{self.description}', example='{self.example}')"


PROMPT_TEMPLATES = [PromptTemplate(
    description="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. Assign the label of yes or no for each given source code comment.",
    example="Comment: {}\nLabel: {}"
)]

In [42]:
# DETECTION_CLASS_MAP = {
#     0: "Not-SATD",
#     1: "SATD"
# }
# DETECTION_CLASS_MAP.update({v: k for k, v in DETECTION_CLASS_MAP.items()})

In [43]:

def create_prompt(prompt_template, x,y, question):
    instances = [prompt_template.example.format(x,y) for _, [x,y] in enumerate(zip(x,y))]
    instances.append(prompt_template.example.format(question, ''))
    return prompt_template.description + "\n" + "\n".join(instances)

In [44]:
MODEL_CONFIGS = [{"name": "Flan T5 Small", "uri": 'google/flan-t5-small'}]


In [45]:
def create_model_and_tokenizer(model_config):
    model = AutoModelForSeq2SeqLM.from_pretrained(model_config['uri']).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_config['uri'])
    return model, tokenizer

def predict_with_prompt(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {key: value.to(device) for key, value in inputs.items()}
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            generation_config = GenerationConfig(max_new_tokens=5, do_sample=True, temperature=0.01)
        )[0],
        skip_special_tokens=True
    )
    return output

In [46]:
FEW_SHOT_SIZES = [0,1,2,3,5,10,15,20]

In [47]:
from enum import Enum

class FewShotSelectionStragegy(Enum):
    RANDOM = 1
    SIMILAR = 2


In [48]:
import random
def pick_n_shot(x,y, index, st_similarity, n = 0, strategy = None):
    if len(x) < n:
        raise Exception(f'only {len(x)} examples available for {n} shots')
    indexes = []
    if strategy == FewShotSelectionStragegy.RANDOM:
        indexes = random.sample(range(len(x)), n)
    elif strategy == FewShotSelectionStragegy.SIMILAR:
        _, top_n_indices = st_similarity[index].topk(n)
        indexes.extend(top_n_indices.tolist())
    shot_x = []
    shot_y = []
    for index in indexes:
        shot_x.append(x[index])
        shot_y.append(y[index])
    return shot_x, shot_y


In [49]:
sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
model_configs = MODEL_CONFIGS[:1]
few_shot_sizes = FEW_SHOT_SIZES[1:2]
prompt_templates = PROMPT_TEMPLATES
few_shot_strategy = FewShotSelectionStragegy.SIMILAR

train_x = dataset["train"]["text"]
train_y = dataset["train"]["label"]

test_x = dataset["test"]["text"]
test_y = dataset["test"]["label"]

#if few_shot_strategy == FewShotSelectionStragegy.SIMILAR:
train_x_st = sentence_transformer.encode(train_x)
test_x_st = sentence_transformer.encode(test_x)
st_similarities = cos_sim(test_x_st, train_x_st)


for few_shot_size in few_shot_sizes:
    for model_config in model_configs:
        model, tokenizer = create_model_and_tokenizer(model_config)
        y_pred = []
        for prompt_template in prompt_templates:
            for  i, [text,label] in enumerate(zip(test_x, test_y)):
                shot_x, shot_y = pick_n_shot(train_x, train_y, i, st_similarities, few_shot_size, few_shot_strategy)
                prompt = create_prompt(prompt_template, shot_x, shot_y, text)
                pred = predict_with_prompt(model, tokenizer, prompt)
                y_pred.append(pred)
            print(set(y_pred))
            print(len(y_pred))
            print(classification_report(test_y, y_pred, zero_division = 0, digits = 3))

    